# Practice Lab: Latency Optimization (Lesson 13.4)

Module 13 . 4 exercises . TTFT + parallel RAG + prefetch + adaptive optimizer


# Lesson 13.4: Latency Optimization

4 exercises: 1E + 2M + 1C

Module 13: Cost Optimization & Efficiency


In [ ]:
import random
random.seed(42)


---
## Exercise 1 (Easy): TTFT Benchmark


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class TTFT:
    def __init__(self,bt=150,pk=30,gm=25): self.bt=bt; self.pk=pk; self.gm=gm
    def sim(self,it,ot=200):
        ttft=max(50,round(self.bt+(it/1000)*self.pk+random.gauss(0,15)))
        total=max(100,round(ttft+ot*self.gm+random.gauss(0,50)))
        return {"it":it,"ot":ot,"ttft":ttft,"total":total,"sp":round(total/max(ttft,1),1)}

b=TTFT()
prompts=[("Hello",50,30),("Simple",100,80),("Short",200,150),("Medium",500,200),("RAG",1000,250),
    ("Long",2000,300),("DocQA",3000,350),("Large",4000,400),("Full",5000,300),("Max",8000,500)]

print("TTFT Benchmark:")
print(f"  {'Name':<8} {'In':>5} {'Out':>5} {'TTFT':>6} {'Total':>7} {'Speedup':>8}")
tbl=[]
for n,i,o in prompts:
    r=b.sim(i,o); tbl.append((i,r["ttft"]))
    print(f"  {n:<8} {i:>5} {o:>5} {r['ttft']:>5}ms {r['total']:>6}ms {r['sp']:>7.1f}x")
print(f"\n  TTFT chart:")
for t,ttft in tbl: print(f"    {t:>5} tok: {ttft:>4}ms {'#'*(ttft//10)}")
avg=sum(t for _,t in tbl)/len(tbl)
print(f"\n  Avg TTFT: {avg:.0f}ms | Rule: ~{b.pk}ms per 1K input tokens")


</details>


---
## Exercise 2 (Medium): Parallel RAG


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class PRAG:
    def __init__(self):
        self.steps={"embed":(50,True),"retrieve":(200,True),"classify":(100,True),
            "rerank":(150,True),"generate":(800,False)}
    def sequential(self):
        t=0
        for n,(ms,_) in self.steps.items(): t+=max(10,round(ms+random.gauss(0,ms*0.1)))
        return t
    def parallel(self,w=5):
        par={n:max(10,round(ms+random.gauss(0,ms*0.1))) for n,(ms,p) in self.steps.items() if p}
        seq=sum(max(10,round(ms+random.gauss(0,ms*0.1))) for n,(ms,p) in self.steps.items() if not p)
        vals=list(par.values())
        pw=0
        for i in range(0,len(vals),w): pw+=max(vals[i:i+w])
        return pw+seq

p=PRAG()
print("Parallel RAG Pipeline:")
print(f"  {'Workers':>8} {'Total':>7} {'Speedup':>8}")
random.seed(42); sq=p.sequential()
for w in [1,2,3,5]:
    random.seed(42); pr=p.parallel(w); sp=sq/max(pr,1)
    print(f"  {w:>8} {pr:>6}ms {sp:>7.1f}x")
print(f"\n  Sequential: {sq}ms | Best parallel: {p.parallel(5)}ms")


</details>


---
## Exercise 3 (Medium): Predictive Prefetch


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class PF:
    def __init__(self,gms=800,cost=0.0002):
        self.gms=gms; self.cost=cost; self.cache={}; self.h=0; self.m=0; self.pf=0
    def prefetch(self,preds):
        for q in preds: self.cache[q.lower().strip()]=True; self.pf+=1
    def query(self,q):
        if q.lower().strip() in self.cache: self.h+=1; return 0
        self.m+=1; return max(100,round(self.gms+random.gauss(0,100)))

pf=PF()
pages={"python":["how much does python cost","python prerequisites","python course duration"],
    "genai":["genai course content","genai course cost","genai projects"],
    "pricing":["discounts available","refund policy","installment payment"],
    "about":["who is instructor","student count","what is netsetos"],
    "ml":["ml tools used","ml course cost","python needed for ml"]}
for _,preds in pages.items(): pf.prefetch(preds)

queries=(["how much does python cost","refund policy","genai course cost","who is instructor",
    "python prerequisites","discounts available","genai course content","ml course cost",
    "installment payment","genai projects","python course duration","python needed for ml",
    "ml tools used","student count","what is netsetos"]+
    ["switch courses","live support","class schedule","job placement","access recordings",
    "cancellation policy","certificate","hours per week","free trial","languages taught",
    "pytorch","cloud deploy","join mid-batch","methodology","corporate training",
    "company age","batch size","reviews","audit course","capstone",
    "devops","cloud platform","community","mentorship","data engineering",
    "contact support","self paced","ide","homework","weekend batches",
    "success rate","system design","demo class","interview prep","alumni"])
random.shuffle(queries); queries=queries[:50]

lats=[pf.query(q) for q in queries]
hr=pf.h/max(pf.h+pf.m,1)*100
avg_w=sum(lats)/len(lats)
print("Predictive Prefetching:")
print(f"  Prefetched: {pf.pf} | Hits: {pf.h}/{pf.h+pf.m} ({hr:.0f}%)")
print(f"  Avg latency: {avg_w:.0f}ms (with) vs {pf.gms}ms (without)")
print(f"  Cost: ${pf.pf*pf.cost:.4f} for {pf.pf} prefetches")
print(f"  Cost/hit: ${pf.pf*pf.cost/max(pf.h,1):.4f}")


</details>


---
## Exercise 4 (Challenge): Adaptive Optimizer


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import random; random.seed(42)

class ALO:
    LEVELS=[{"n":"normal","th":3000,"m":1.0},{"n":"cap_output","th":5000,"m":0.6},
        {"n":"flash","th":10000,"m":0.3},{"n":"queue","th":999999,"m":0.1}]
    def __init__(self,w=20):
        self.w=w; self.lats=[]; self.lv=0; self.hist=[]; self.adaps=[]
    def p95(self):
        if len(self.lats)<5: return 0
        r=sorted(self.lats[-self.w:]); return r[int(len(r)*0.95)]
    def process(self,raw):
        actual=max(50,round(raw*self.LEVELS[self.lv]["m"]+random.gauss(0,50)))
        self.lats.append(actual); p=self.p95(); old=self.lv
        if p>10000 and self.lv<3: self.lv=3
        elif p>5000 and self.lv<2: self.lv=2
        elif p>3000 and self.lv<1: self.lv=1
        elif p<2000 and self.lv>0: self.lv=max(0,self.lv-1)
        if old!=self.lv:
            self.adaps.append({"r":len(self.lats),"f":self.LEVELS[old]["n"],
                "t":self.LEVELS[self.lv]["n"],"p95":round(p)})
        self.hist.append({"lat":actual,"p95":round(p),"lv":self.LEVELS[self.lv]["n"]})
        return actual

traffic=[]
for i in range(100):
    if i<30: traffic.append(max(200,round(random.gauss(1500,300))))
    elif i<50: traffic.append(max(200,round(random.gauss(6000,1500))))
    elif i<70: traffic.append(max(200,round(random.gauss(12000,2000))))
    else: traffic.append(max(200,round(random.gauss(2000,500))))

a=ALO(20)
adapted=[a.process(t) for t in traffic]

print("Adaptive Latency Optimizer:")
print(f"  {'Req':>4} {'Raw':>7} {'Adapted':>8} {'P95':>7} {'Level':>12}")
for i in [0,15,30,40,50,60,70,85,99]:
    h=a.hist[i]
    print(f"  {i+1:>4} {traffic[i]:>6}ms {h['lat']:>7}ms {h['p95']:>6}ms {h['lv']:>12}")

print(f"\n  Adaptations: {len(a.adaps)}")
for ad in a.adaps: print(f"    Req {ad['r']:>3}: {ad['f']}->{ad['t']} (P95={ad['p95']}ms)")

def avg(l): return sum(l)//max(len(l),1)
def p95(l): s=sorted(l); return s[int(len(s)*0.95)] if s else 0
print(f"\n  No-opt avg={avg(traffic)}ms P95={p95(traffic)}ms")
print(f"  Adaptive avg={avg(adapted)}ms P95={p95(adapted)}ms")
print(f"  Improvement: avg {(1-avg(adapted)/max(avg(traffic),1))*100:.0f}%"
      f" P95 {(1-p95(adapted)/max(p95(traffic),1))*100:.0f}%")


</details>
